In [5]:
import openpyxl as xl
from openpyxl import Workbook as wb

In [48]:
demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
    keep_vba = True, # detect vba
    rich_text = True, # detect rich text in cells
    data_only = False, # preserve formulas not their values
    # note read_only, which uses less memory but lacks charts and images
    # note keep_links which keeps cache of data stored from external wb
)

# Merged Cells

We're going to check if each cell has been merged.  This turned out to be easier than I thought because there's a dedicated type for it.

In [57]:
# Load the demo workbook, sheet about merged cells
demo_ws = demo_book['merged_cells']

# For each column in the sheet
for col in demo_ws.columns:

    # For each cell in that column
    for cell in col:

        # If the cell has been merged to another cell
        if isinstance( cell, xl.cell.cell.MergedCell ):
           
           # Get the location of the cell
           print(cell.coordinate)

print("correct is C3")

C3
correct is C3


This concept can be extended to collect all the coordinates of every merged cell and dump them in a file or other output, such as mongodb.  right now, all but the top left cell in the merged range is detected.  Edge case, when the merged range is size two you can't tell whether the merger is with the cell on the left or the cell above.

Could be useful to know the leftmost cell.  Also, return as ranges of merged cells instead of individual cells

# Detect external references
We're going to look for Array Fo;mulas with brackets in their definitions

In [48]:
# Need regular expressions to search for brackets
import re

demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
    data_only = False, # preserve formulas not their values
)


# Load the demo workbook, sheet about external references
demo_ws = demo_book['external_ref']


# Iterate through columns
for col in demo_ws.columns:

    # Iterate through cells in that column
    for cell in col:

        # Check whether the cell contains a formula
        # This might fail if the cell contains text that starts
        # with equals
        if str(cell.internal_value).startswith("="):

            # Create search pattern for external refs
            # \[ and \] match brackets, \ needed for escape
            # . matches any character
            # * matches any num of previous
            pattern = r'\[.*\]'

            # Check whether formula contains external ref
            if bool(  re.search(pattern, cell.internal_value)  ):

                # Geth the location of the external ref
                print(cell.coordinate)

print("correct is B2:B5")

B2
B3
B4
B5
correct is B2:B5


The test is successful, but I need to confirm whether there are other patterns for formulae.

# Space between data and edges of grid
Check for blank first row or first column 

In [58]:
demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
)


# Load the demo workbook, sheet about external references
demo_ws = demo_book['extra_space']



# Iterate through columns
for idx, col in enumerate(demo_ws.columns):


    # Check whether this entire column is empty
    # map visits each cell and checks whether contents empty
    # all checks whether every single cell empty
    if all( map(lambda cell : cell.internal_value == None, col)   ):

        # Keep looking for the first nonempty
        continue

    else:   

        # This column has nonempty cells, and it is the first
        # Print the index of this col
        print(f"first nonempty col is {idx}")
        break

# Iterate through rows
for idx, row in enumerate(demo_ws.rows):

    # Check whether this entire column is empty
    # map applies lambda func to each cell
    # lambda func checks whether contents empty 
    # all checks whether every single cell empty
    if all( map(lambda cell : cell.internal_value == None, row)   ):

        # Keep looking for the first nonempty
        continue

    else:   

        # This column has nonempty cells, and it is the first
        # Print the index of this row
        print(f"first nonempty row is {idx}")

        # End the loop
        break

print("correct is column 2 and row 1")

first nonempty col is 2
first nonempty row is 1
correct is column 2 and row 1


The test is successful.  This implementation will fail if there are other accidental or invisible entities attached to cells on the edges.  A memory conservative, generator based approach is possible with the toolz package.  It may be desireable to create an iterator that outputs the excel letter based system for indexing the columns, but I will leave that aside for now because I know it's possible.

# Column headers are variable names

if column headers, headers are 3–12 characters, start with a letter (not number) and contain no spaces.

In [18]:
demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
)


# Load the demo workbook, sheet about external references
demo_ws = demo_book['headers']

def is_valid(header):
    validity = isinstance(header, str)
    validity = validity and (len(header) >= 3 and len(header) <= 12)
    validity = validity and not header[0].isdigit()
    validity = validity and not (" " in header)
    return validity

bad_headers = dict()
good_headers = dict()
for idx, col in enumerate(demo_ws.iter_cols(max_row = 1, values_only = True)):
    header = col[0]
    if is_valid(header):
        good_headers[idx] = header
    else:
        bad_headers[idx] = header
print(good_headers)
print(bad_headers)


{4: 'valid', 5: 'valid2'}
{0: 'This_header_is_too_long', 1: 'xx', 2: '3starts_number', 3: 'has space', 6: None}


The test is successful for the given heuristic.  It will only handle cases where the headers are on the first row.

# Detect special characters in cells and or white space

no leading/trailing spaces or special characters $, @, %, #, &, *, (, ), !, /.  Cells that only contain a question mark.

In [38]:
import re

demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
)


# Load the demo workbook, sheet about external references
demo_ws = demo_book['special']

def is_valid(cell):

    value = cell.internal_value

    if value == None:
        return True

    # Check cell contains non word characters
    if bool(  re.search(r'[^a-zA-Z?\d\s]', value)  ):

        return False

    # Check cell has no leading or trailing whitespace
    if bool( re.search(r'^\s|\s$', value) ):

        return False
    
    # Check cell isn't a question mark
    if value == "?":

        return False
    
    # If pass all above, return true
    return True
        




bad_cells = dict()
good_cells = dict()
for idx, col in enumerate(demo_ws.iter_cols()):
    
    for cell in col:
        if is_valid(cell):
            good_cells[cell.coordinate] = cell.internal_value
        else:
            bad_cells[cell.coordinate] = cell.internal_value
print(good_cells)
print(bad_cells)

{'A1': 'header1', 'A8': None, 'A9': None, 'A10': None, 'A11': None, 'A12': None, 'A13': None, 'A14': None, 'A15': None, 'A16': None, 'A17': None, 'A18': None, 'A19': None, 'A20': None, 'A21': None, 'A22': None, 'A23': None, 'A24': None, 'A25': None, 'A26': None, 'A27': 'okay', 'B1': 'header2', 'B7': 'just fine', 'B8': 'question?', 'B9': None, 'B10': None, 'B11': None, 'B12': None, 'B13': None, 'B14': None, 'B15': None, 'B16': None, 'B17': None, 'B18': None, 'B19': None, 'B20': None, 'B21': None, 'B22': None, 'B23': None, 'B24': None, 'B25': None, 'B26': None, 'B27': None}
{'A2': 'bad!', 'A3': 'n@ope', 'A4': 'sorry%', 'A5': 'uh#uh', 'A6': 'space ', 'A7': ' space', 'B2': 'try&again', 'B3': 'still*no', 'B4': '(fail', 'B5': 'wron)g', 'B6': 'ne/ver'}


Misses punctuation other than ? in a text string, diacritic marks, needs to exception headers, punctuation alone.  Correct punctuation: .,-_:;?!/'".  Check whether hyperlink cell contents refers to visible text or link or both.

# Detect images

As stated.  Is there an image anywhere in this workbook?

In [49]:
demo_book = xl.load_workbook(
    "data/demo.xlsx", # file to load
)


# Load the demo workbook, sheet about external references
demo_ws = demo_book['images']

if hasattr(demo_ws, "_images") and demo_ws._images:
    img_anchors = [img.anchor for img in demo_ws._images]
else:
    print("no images")

no images


Not working.  I'm not sure why the images I placed on the sheet are not showing up.  Not high enough priority to pursue extra-python solutions.

# Annotation or subheader
Rows of mostly empty values. Look for cells with a lot of characters

# Comment columns

Columns with headers containing any of: comment, note, remark, observation, annotation, memo, discussion

In [ ]:
demo_book = xl.load_workbook(
    "data/demo.xlsx"
)

# Load the demo workbook, sheet about comments
demo_ws = demo_book['comment_col']

# Convert one letter into regex for upper and lower case
def create_regex(word):

    # Initialize empty regex
    regex = ""

    # loop through each letter
    for letter in word:
        
        # append a regex pattern to match both up/low case
        regex += f"[{letter.lower()}{letter.upper()} ]"

    # return the regex
    return regex

# Initialize dict to hold comment columns
comment_cols = dict()

# Loop over columns, col is a tuple
for col in demo_ws.iter_cols(max_row = 1):

    # Get first and only cell
    cell = col[0]

    # Get string stored in cell
    header = cell.internal_value

    # Words to check for in the header
    # Matches plural, or any header that contains one of
    # these words.  e.g. Draft annotations
    searches = [
            "comment",
            "note",
            "remark",
            "observation",
            "annotation",
            "memo",
            "discussion",
            "other"]

    # convert searches into regex patterns that match case variation  
    search_expand = [create_regex(word) for word in searches]

    # loop over the regex's
    for idx, regex in enumerate(search_expand):

        # if header contains variant of regex
        if re.search(regex, header):

            # collect this cell's coordinate, and return the header
            comment_cols[cell.coordinate] = header, searches[idx]


comment_cols
    

{'C1': ('Comments', 'comment'),
 'D1': ('noted', 'note'),
 'E1': ('annoTation', 'annotation')}